## Población 2010
En este notebook procesaremos 52 jSON, uno por cada provincia y ciudad autónoma.  
Cada uno de ellos contiene el total de población por municipios así como la segmentación por sexo.  
Obtendremos un Dataframe con la siguiente información:
- Y2010
- Nombre Provincia
- Codigo Provincia
- Nombre Municipio
- Codigo Municipio
- Poblacion Total
- Poblacion Hombres
- Poblacion Mujeres

In [1]:
import pandas as pd
import numpy as np
import requests

Creamos una lista con el número de tabla correspondiente a cada provincia y ciudad autónoma:

In [2]:
CodTablas = [2854, 2855, 2856, 2857, 2858, 2859, 2860, 2861, 2862, 2863, 2864, 2865, 2866, 2868, 2869, 2870,2871,
2872, 2873,  2874, 2875, 2876, 2877, 2878, 2879, 2880, 2881, 2882, 2883, 2884, 2885, 2886, 2888, 2889, 2890, 2891, 
2892, 2893, 2894, 2895, 2896, 2899, 2900, 2901,2902, 2903, 2904, 2905, 2906, 2907,  2908, 2909]

In [3]:
def ine_request(ine_code):
    path_template = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/{cod_serie}?tip=AM&'
    path = path_template.format(cod_serie=ine_code)
    json_request = requests.get(path).json()
    
    return json_request

In [4]:
AnyoHomList = list()
CodHomList = list()
PobHomList = list()
PobMujList = list()
AnyoMujList = list()
CodMujList = list()
PobTotList = list()
AnyoTolList = list()
CodTolList = list()

Nos quedamos solo con el año 2010:

In [5]:
for codigo in CodTablas:
    datos = ine_request(codigo)
    for dato in datos: 
        for i in range(0,len(dato['Data'])):
            if  dato['Data'][i]['Anyo'] == 2010 and 'Hombres' in dato['Nombre']:
                PobHomList.append(dato['Data'][i]['Valor'])
                AnyoHomList.append(dato['Data'][i]['Anyo'])
                CodHomList.append(dato['MetaData'][0]['Codigo'])
            elif dato['Data'][i]['Anyo'] == 2010 and 'Mujeres' in dato['Nombre']:
                PobMujList.append(dato['Data'][i]['Valor'])
                AnyoMujList.append(dato['Data'][i]['Anyo'])
                CodMujList.append(dato['MetaData'][0]['Codigo'])
            elif dato['Data'][i]['Anyo'] == 2010 and ('Hombres'and'Mujeres') not in dato['Nombre']:
                PobTotList.append(dato['Data'][i]['Valor'])
                AnyoTolList.append(dato['Data'][i]['Anyo'])
                CodTolList.append(dato['MetaData'][0]['Codigo'])            

In [6]:
PoblacionHombres = pd.DataFrame({
        'Y2010' : AnyoHomList,
        'Codigo Municipio' : CodHomList,
        'Poblacion Hombre': PobHomList
    })
PoblacionMujeres = pd.DataFrame({
        'Y2010' : AnyoMujList,
        'Codigo Municipio' : CodMujList,
        'Poblacion Mujer': PobMujList
    })
PoblacionTotal = pd.DataFrame({
        'Y2010' : AnyoTolList,
        'Codigo Municipio' : CodTolList,
        'Poblacion Total': PobTotList
    })

In [7]:
Poblacion10 = pd.concat([PoblacionTotal, PoblacionHombres, PoblacionMujeres], axis = 1)
Poblacion10 = Poblacion10.loc[:, ~Poblacion10.columns.duplicated()]

Cargamos la tabla 01_Output_ProvMun10.xlsx para posteriormente realizar el merge con Poblacion10:

In [8]:
ProvMun = pd.read_excel('01_Output_ProvMun10.xlsx', dtype='object') 
ProvMun.head()

,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio
0,Araba/Álava,01,Agurain/Salvatierra,01051
1,Araba/Álava,01,Alegría-Dulantzi,01001
2,Araba/Álava,01,Amurrio,01002
3,Araba/Álava,01,Añana,01049
4,Araba/Álava,01,Aramaio,01003


In [9]:
Poblacion10 = Poblacion10.merge(ProvMun, on='Codigo Municipio')

In [10]:
Poblacion10.head()

,Y2010,Codigo Municipio,Poblacion Total,Poblacion Hombre,Poblacion Mujer,Nombre Provincia,Codigo Provincia,Nombre Municipio
0,2010,01001,2714.0,1395.0,1319.0,Araba/Álava,01,Alegría-Dulantzi
1,2010,01002,10050.0,5047.0,5003.0,Araba/Álava,01,Amurrio
2,2010,01049,175.0,92.0,83.0,Araba/Álava,01,Añana
3,2010,01003,1497.0,790.0,707.0,Araba/Álava,01,Aramaio
4,2010,01006,225.0,112.0,113.0,Araba/Álava,01,Armiñón


In [11]:
ColNue = ['Y2010','Nombre Provincia', 'Codigo Provincia', 'Nombre Municipio',
         'Codigo Municipio', 'Poblacion Total', 'Poblacion Hombre', 'Poblacion Mujer']
Poblacion10 = Poblacion10[ColNue]

In [12]:
Poblacion10.head()

,Y2010,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Poblacion Total,Poblacion Hombre,Poblacion Mujer
0,2010,Araba/Álava,01,Alegría-Dulantzi,01001,2714.0,1395.0,1319.0
1,2010,Araba/Álava,01,Amurrio,01002,10050.0,5047.0,5003.0
2,2010,Araba/Álava,01,Añana,01049,175.0,92.0,83.0
3,2010,Araba/Álava,01,Aramaio,01003,1497.0,790.0,707.0
4,2010,Araba/Álava,01,Armiñón,01006,225.0,112.0,113.0


In [13]:
Poblacion10.to_excel('02_Output_Poblacion10.xlsx', header = True, index = False)